## 准备数据

In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, optimizers, datasets

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # or any {'0', '1', '2'}

def mnist_dataset():
    (x, y), (x_test, y_test) = datasets.mnist.load_data()
    #normalize
    x = x/255.0
    x_test = x_test/255.0
    
    return (x, y), (x_test, y_test)

In [ ]:
print(list(zip([1, 2, 3, 4], ['a', 'b', 'c', 'd'])))

[(1, 'a'), (2, 'b'), (3, 'c'), (4, 'd')]


## 建立模型

In [ ]:
class myModel:
    def __init__(self):
        ####################
        '''声明模型对应的参数'''
        ####################
        self.W1 = tf.Variable(tf.random.normal([784, 512], stddev=0.1))
        self.b1 = tf.Variable(tf.zeros([512]))
        self.W2 = tf.Variable(tf.random.normal([512, 10], stddev=0.1))
        self.b2 = tf.Variable(tf.zeros([10]))
        
    def __call__(self, x):
        ####################
        '''实现模型函数体，返回未归一化的logits'''
        ####################
        x = tf.reshape(x, [-1, 784])
        h = tf.nn.relu(tf.matmul(x, self.W1) + self.b1)
        logits = tf.matmul(h, self.W2) + self.b2
        return logits
        
model = myModel()

optimizer = optimizers.Adam()

## 计算 loss

In [ ]:
@tf.function
def compute_loss(logits, labels):
    return tf.reduce_mean(
        tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels))

@tf.function
def compute_accuracy(logits, labels):
    predictions = tf.argmax(logits, axis=1)
    return tf.reduce_mean(tf.cast(tf.equal(predictions, labels), tf.float32))

@tf.function
def train_one_step(model, optimizer, x, y):
    with tf.GradientTape() as tape:
        logits = model(x)
        loss = compute_loss(logits, y)

    # compute gradient
    trainable_vars = [model.W1, model.W2, model.b1, model.b2]
    grads = tape.gradient(loss, trainable_vars)
    for g, v in zip(grads, trainable_vars):
        v.assign_sub(0.01*g)

    accuracy = compute_accuracy(logits, y)

    # loss and accuracy is scalar tensor
    return loss, accuracy

@tf.function
def test(model, x, y):
    logits = model(x)
    loss = compute_loss(logits, y)
    accuracy = compute_accuracy(logits, y)
    return loss, accuracy

## 实际训练

In [ ]:
train_data, test_data = mnist_dataset()
for epoch in range(50):
    loss, accuracy = train_one_step(model, optimizer, 
                                    tf.constant(train_data[0], dtype=tf.float32), 
                                    tf.constant(train_data[1], dtype=tf.int64))
    print('epoch', epoch, ': loss', loss.numpy(), '; accuracy', accuracy.numpy())
loss, accuracy = test(model, 
                      tf.constant(test_data[0], dtype=tf.float32), 
                      tf.constant(test_data[1], dtype=tf.int64))

print('test loss', loss.numpy(), '; accuracy', accuracy.numpy())

epoch 0 : loss 0.82007736 ; accuracy 0.7744833
epoch 1 : loss 0.8173891 ; accuracy 0.77543336
epoch 2 : loss 0.8147299 ; accuracy 0.77615
epoch 3 : loss 0.8120997 ; accuracy 0.7768667
epoch 4 : loss 0.8094978 ; accuracy 0.77755
epoch 5 : loss 0.80692387 ; accuracy 0.7780667
epoch 6 : loss 0.80437744 ; accuracy 0.77898335
epoch 7 : loss 0.8018578 ; accuracy 0.77968335
epoch 8 : loss 0.7993648 ; accuracy 0.7804833
epoch 9 : loss 0.7968979 ; accuracy 0.78111666
epoch 10 : loss 0.79445666 ; accuracy 0.78173333
epoch 11 : loss 0.79204035 ; accuracy 0.78246665
epoch 12 : loss 0.78964883 ; accuracy 0.7833
epoch 13 : loss 0.78728163 ; accuracy 0.7837667
epoch 14 : loss 0.7849384 ; accuracy 0.7845
epoch 15 : loss 0.78261876 ; accuracy 0.7852
epoch 16 : loss 0.78032243 ; accuracy 0.7858
epoch 17 : loss 0.77804905 ; accuracy 0.78651667
epoch 18 : loss 0.7757983 ; accuracy 0.78713334
epoch 19 : loss 0.7735699 ; accuracy 0.7876
epoch 20 : loss 0.77136326 ; accuracy 0.7884833
epoch 21 : loss 0.76917